In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import PolynomialFeatures
from umap import UMAP

from climate_attitudes.datasets.reduced_no_imputation import schema
from climate_attitudes.visualisation import DIVERGING_CMAP, configure_mpl
from ising import Ising

RANDOM_SEED = 202606041925

rng = np.random.default_rng(RANDOM_SEED)


np.set_printoptions(linewidth=200)

configure_mpl(Path("../fonts"))

DATA_PATH = Path("../reports/thesis/results/data/model/all_interventions/")

schema = schema.post_index()

In [ ]:
labels = np.load(DATA_PATH / "ising_00_no_use_covariates.npz")["labels"]

inits = np.load(DATA_PATH / "ising_00_no_use_covariates.npz")["measurements"][
    0, :, 0, 0
]
inits_orig = np.load(DATA_PATH / "ising_00_no_use_covariates.npz")["Y0"][:, 1]

no_int_asym_outcome = np.load(DATA_PATH / "ising_00_no_use_covariates.npz")[
    "measurements"
][:, :, 5]
int_asym_outcome = np.load(DATA_PATH / "ising_15_no_use_covariates.npz")[
    "measurements"
][:, :, 5]

int_effect = int_asym_outcome - no_int_asym_outcome

individual_int_effect_on_policy = int_effect[..., 7].mean(axis=0)

In [ ]:
inits.shape

In [ ]:
individual_int_effect_on_policy

Looking at CC worry (idx 2) and climate impacts (idx 6)

In [ ]:
individual_effect_ccw_ccp = individual_int_effect_on_policy[:, 2]
individual_effect_cci_ccp = individual_int_effect_on_policy[:, 6]

In [ ]:
def get_percentile_idxes(a, plim):
    vlow, vhigh = np.percentile(a, plim)
    print(vlow, vhigh)
    return np.argwhere((a >= vlow) & (a <= vhigh)).flatten()

In [ ]:
ccw_ccp_lo_eff_idxes = get_percentile_idxes(individual_effect_ccw_ccp, (0, 10))
ccw_ccp_hi_eff_idxes = get_percentile_idxes(individual_effect_ccw_ccp, (90, 100))

cci_ccp_lo_eff_idxes = get_percentile_idxes(individual_effect_cci_ccp, (0, 10))
cci_ccp_hi_eff_idxes = get_percentile_idxes(individual_effect_cci_ccp, (90, 100))

In [ ]:
individual_effect_cci_ccp.shape

Check overlap --- 32%.

In [ ]:
(
    len(set(ccw_ccp_lo_eff_idxes) & set(cci_ccp_lo_eff_idxes))
    / len(set(ccw_ccp_lo_eff_idxes) | set(cci_ccp_lo_eff_idxes))
)

For a given set of individuals, calculate the probability that each initial spin state is 1

In [ ]:
def init_spin_probs(idxes, inits):
    print((inits[idxes] == np.ones(inits.shape[-1])).mean(axis=0))

In [ ]:
init_spin_probs(ccw_ccp_lo_eff_idxes, inits)

In [ ]:
init_spin_probs(ccw_ccp_hi_eff_idxes, inits)

In [ ]:
labels

We can start with the regression with interactions, fit directly on the continuous outcome (maybe no need to binarize, since keeping the outcome continuous preserves more of the signal? Or we have all versions of the binarization as the single individual). The nice thing is we don’t have to decide the interaction terms ourselves: we generate the pairwise interactions (polynomial features, interaction-only) and let LASSO with cross-validation keep the ones that matter and zero out the rest. The signs of the surviving coefficients then give us the direction of each feature (higher X1 gives bigger effect, and so on). An alternative that I found online, but I don’t know much about is an EBM (explainable boosting machine), which lets us “see” the response curve for each feature and the main pairwise interactions by plotting them. Same goal either way: a readable, signed picture of how each feature moves the outcome.

## Regression with interactions

Fit regression for expected effect of intervention on $X$, where $X$ is either:
- The initial states (original, unbinarised data), or
- The initial effective local fields experienced at each spin.

Include (or maybe only use) pairwise interaction terms

In [ ]:
preproc = PolynomialFeatures()
X = preproc.fit_transform(inits_orig)

For 8 spins, $X$ has 45 features:

- 1 intercept term
- 8 degree-1 features
- 8 self-interaction degree-2 features
- 28 = 8*7/2 pairwise interaction features (excluding duplicates)

In [ ]:
Y = individual_effect_ccw_ccp

lasso_cv = LassoCV(cv=10).fit(X, Y)

In [ ]:
degree_1_features = lasso_cv.coef_[1:9]
degree_2_features = np.zeros((8, 8), dtype=np.float64)
degree_2_features[np.triu_indices_from(degree_2_features)] = lasso_cv.coef_[9:]

degree_1_features[abs(degree_1_features) < 1e-4] = 0
degree_2_features[abs(degree_2_features) < 1e-4] = 0

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 2), constrained_layout=True)
sns.barplot(degree_1_features, ax=ax)
ax.set_xticks(np.arange(8), labels, rotation=45, horizontalalignment="right");

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5.5), constrained_layout=True)
ax.set_aspect("equal")
mask = np.triu(np.ones((8, 8), dtype=np.bool), k=1)[:, ::-1]
sns.heatmap(
    degree_2_features[::-1],
    mask=mask,
    vmin=-0.05,
    vmax=0.05,
    center=0,
    cmap=DIVERGING_CMAP,
    annot=True,
    fmt=".2f",
    linewidth=0.5,
    cbar_kws=dict(shrink=0.65, aspect=25),
    ax=ax,
)
ax.set_yticks(np.arange(8) + 0.5, reversed(labels), rotation=0)
ax.set_xticks(np.arange(8) + 0.5, labels, rotation=35, horizontalalignment="right");

Investigating why deg-1 for cc-worry is negative, while deg-2 is positive.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3), constrained_layout=True)
sns.violinplot(x=inits_orig[:, 2] ** 2, y=Y);

Low effect common for the interaction, indicating that the intervention is less effective for individuals who are either _very unconcerned_ or _very concerned_. This aligns with our expectations, as the intervention has a smaller effect for these individuals. Those who are very unconcerned require a larger intervention to change their worry level. Those who are very worried cannot increase their worry level much.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3), constrained_layout=True)
sns.violinplot(x=inits_orig[:, 2], y=Y);

Indeed, we see this also in the second-order self-interactions. Individuals who are very worried are least affected by the intervention. Those who are very unconcerned are _more_ affected, but less so than those who are only slightly worried. Those who are already somewhat worried exhibit high variance but a small average effect.

We can also do the same thing, but instead looking at the effective local field at each spin at $t=0$. This tells us what the existing pressure was like for each individual prior to intervention.

For each replicate, each individual, and each repeat, calculate the effective local field. Average this over repeats to estimate the expected effective local field for each individual in the replicate. 

Then regress the average effect for each individual within each replicate on the local fields.

In [ ]:
inits

In [ ]:
inits_orig

In [ ]:
res_00 = np.load(DATA_PATH / "ising_00_no_use_covariates.npz")
res_15 = np.load(DATA_PATH / "ising_15_no_use_covariates.npz")
measurements_00 = res_00["measurements"][:, :, 2, 7]
measurements_15 = res_15["measurements"][:, :, 2, 7]
params = res_00["params"]
inits = res_00["measurements"][:, :, 0, 2]

X = np.array([1.0])

repeats, individuals, _ = measurements_00.shape
# repeats = 1

adj = np.ones((8, 8), dtype=bool)

eff_local_fields = np.empty((repeats, individuals, 8), dtype=np.float64)
final_eff_local_field_null = np.empty((repeats, individuals), dtype=np.float64)
final_eff_local_field_int = np.empty((repeats, individuals), dtype=np.float64)
for repeat in range(repeats):
    p = params[repeat]
    h = p[:8]
    j = p[8:].reshape((8, 8))

    for individual in range(individuals):
        y0 = inits[repeat, individual]
        eff_local_fields[repeat, individual] = Ising.parallel_glauber_theta(
            y0, X, h, j, adj
        )
        final_eff_local_field_null[repeat, individual] = Ising.parallel_glauber_theta(
            measurements_00[repeat, individual], X, h, j, adj
        )[7]
        final_eff_local_field_int[repeat, individual] = Ising.parallel_glauber_theta(
            measurements_15[repeat, individual], X, h, j, adj
        )[7]

eff_local_fields = eff_local_fields.mean(axis=0)
final_eff_local_field = (final_eff_local_field_int - final_eff_local_field_null).mean(
    axis=0
)

In [ ]:
log_p_int = final_eff_local_field_int.mean(axis=0) - np.log(
    2 * np.cosh(final_eff_local_field_int.mean(axis=0))
)
log_p_null = final_eff_local_field_null.mean(axis=0) - np.log(
    2 * np.cosh(final_eff_local_field_null.mean(axis=0))
)
log_odds = log_p_int - log_p_null

In [ ]:
preproc = PolynomialFeatures()
X = preproc.fit_transform(eff_local_fields)
Y = final_eff_local_field.flatten()
Y = log_p_int

lasso_cv = LassoCV(cv=10, max_iter=10_000).fit(X, Y)

In [ ]:
final_eff_local_field.shape

In [ ]:
degree_1_features = lasso_cv.coef_[1:9]
degree_2_features = np.zeros((8, 8), dtype=np.float64)
degree_2_features[np.triu_indices_from(degree_2_features)] = lasso_cv.coef_[9:]

degree_1_features[abs(degree_1_features) < 1e-4] = 0
degree_2_features[abs(degree_2_features) < 1e-4] = 0

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 2), constrained_layout=True)
sns.barplot(degree_1_features, ax=ax)
ax.set_xticks(np.arange(8), labels, rotation=45, horizontalalignment="right");

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5.5), constrained_layout=True)
ax.set_aspect("equal")
mask = np.triu(np.ones((8, 8), dtype=np.bool), k=1)[:, ::-1]
sns.heatmap(
    degree_2_features[::-1],
    mask=mask,
    vmin=-0.05,
    vmax=0.05,
    center=0,
    cmap=DIVERGING_CMAP,
    annot=True,
    fmt=".2f",
    linewidth=0.5,
    cbar_kws=dict(shrink=0.65, aspect=25),
    ax=ax,
)
ax.set_yticks(np.arange(8) + 0.5, reversed(labels), rotation=0)
ax.set_xticks(np.arange(8) + 0.5, labels, rotation=35, horizontalalignment="right");

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3), constrained_layout=True)
sns.scatterplot(x=X[:, 2], y=Y);

## Who is 'CC Worry' intervention ineffective for?

In [ ]:
umap = UMAP()

X = np.concat((inits_orig[ccw_ccp_lo_eff_idxes], (inits_orig[ccw_ccp_hi_eff_idxes])))
Y = np.concat((np.zeros(ccw_ccp_lo_eff_idxes.size), np.ones(ccw_ccp_hi_eff_idxes.size)))

embs_ccw_ccp = umap.fit_transform(X, y=Y)

# embs_lo_ccw_ccp = umap.fit_transform(inits_orig[ccw_ccp_lo_eff_idxes])
# embs_hi_ccw_ccp = umap.fit_transform(inits_orig[ccw_ccp_hi_eff_idxes])

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=6)
colours = kmeans.fit_predict(embs_ccw_ccp)

plt.scatter(embs_ccw_ccp[:, 0], embs_ccw_ccp[:, 1], c=colours, s=3)
plt.colorbar()

In [ ]:
labels

Three clusters for effective:

- CC not human caused
- Not worried
- 

In [ ]:
from scipy.cluster.hierarchy import dendrogram, ward

In [ ]:
fig, ax = plt.subplots()
dendrogram(ward(inits_orig[ccw_ccp_lo_eff_idxes].T), labels=labels)
ax.tick_params(axis="x", rotation=90)

In [ ]:
eff = [0, 2, 5]
print(X[colours == 0].mean(axis=0))
print(X[colours == 2].mean(axis=0))
print(X[colours == 5].mean(axis=0))

In [ ]:
plt.scatter(embs_ccw_ccp[:, 0], embs_ccw_ccp[:, 1], c=Y, s=3)

In [ ]:
colours = inits_orig[ccw_ccp_hi_eff_idxes].mean(axis=1)
plt.scatter(embs_ccw_ccp[:, 0], embs_ccw_ccp[:, 1], c=colours, s=5, cmap="Spectral")
plt.gca().set_aspect("equal", "datalim")
plt.colorbar()

In [ ]:
labels

In [ ]:
colours = inits_orig[ccw_ccp_hi_eff_idxes][:, 7] > 0
plt.scatter(
    embs_ccw_ccp[:, 0],
    embs_ccw_ccp[:, 1],
    c=colours,
    s=5,
    cmap="Spectral",
    vmin=0,
    vmax=1,
)
plt.gca().set_aspect("equal", "datalim")
plt.colorbar()